# Bias Correction (Russia)
**RCP 4.5 | EQM | CRU-TS v4.08 | 1976–2005 baseline | 2041–2070 future**

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import PchipInterpolator
from scipy.ndimage import zoom as _ndz
import os
import warnings
warnings.filterwarnings('ignore')

from dask.distributed import Client
client = Client(n_workers=2, threads_per_worker=2, memory_limit='3GB')
print(client)

plt.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 10, 'figure.titlesize': 13,
    'contour.negative_linestyle': 'solid',
})
%matplotlib inline
print('Libraries imported successfully.')


## Step 1: Configuration

In [ ]:
BASE_DIR    = r'[PROJECT_DIR]'
CRU_FILE    = r'[]\\cru_ts4.08.1901.2023.pre.dat.nc'
PROJECT_DIR = r'[PROJECT_DIR]'
BC_DIR      = os.path.join(PROJECT_DIR, 'data', 'bias_corrected', 'Russia')
ENS_DIR     = os.path.join(PROJECT_DIR, 'data', 'multi_model_ensemble')
os.makedirs(BC_DIR,  exist_ok=True)
os.makedirs(ENS_DIR, exist_ok=True)

RUSSIA_BOUNDS = {'lon': slice(19, 192), 'lat': slice(41, 82)}
RUSSIA_EXTENT = [19, 192, 40, 83]
RUSSIA_PROJ   = ccrs.PlateCarree(central_longitude=105)

BC_START = '1976'; BC_END = '2005'
FUT_START = '2041'; FUT_END = '2070'
N_QUANTILES = 99

models = {
    'HadGEM2-AO': {
        'historical': os.path.join(BASE_DIR, 'pr_day_HadGEM2-AO_historical_r1i1p1_18600101-20051230.nc'),
        'rcp45':      os.path.join(BASE_DIR, 'pr_day_HadGEM2-AO_rcp45_r1i1p1_20060101-21001230.nc'),
        'calendar':   '360_day',
    },
    'MPI-ESM-MR': {
        'historical': [
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_historical_r1i1p1_19700101-19791231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_historical_r1i1p1_19800101-19891231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_historical_r1i1p1_19900101-19991231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_historical_r1i1p1_20000101-20051231.nc'),
        ],
        'rcp45': [
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_rcp45_r1i1p1_20400101-20491231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_rcp45_r1i1p1_20500101-20591231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_rcp45_r1i1p1_20600101-20691231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_rcp45_r1i1p1_20700101-20791231.nc'),
        ],
        'calendar': 'standard',
    },
    'IPSL-CM5A-LR': {
        'historical': os.path.join(BASE_DIR, 'pr_day_IPSL-CM5A-LR_historical_r1i1p1_19500101-20051231.nc'),
        'rcp45':      os.path.join(BASE_DIR, 'pr_day_IPSL-CM5A-LR_rcp45_r1i1p1_20060101-22051231.nc'),
        'calendar':   '360_day',
    },
}
print('Configuration ready.')
print(f'  Baseline: {BC_START}–{BC_END}  |  Future: {FUT_START}–{FUT_END}')


## Step 2: Visualisation helpers

In [ ]:
_OCEAN = cfeature.NaturalEarthFeature(
    'physical', 'ocean', '10m', facecolor='white', edgecolor='none')
_LAKES = cfeature.NaturalEarthFeature(
    'physical', 'lakes', '10m', facecolor='white', edgecolor='none')

def _upsample(data, factor, order):
    mask = np.isnan(data)
    filled = np.where(mask, 0.0, data)
    up = _ndz(filled, factor, order=order)
    mask_up = _ndz(mask.astype(float), factor, order=0) > 0.5
    up[mask_up] = np.nan
    return up

def _upsample(data, factor, order):
    filled = np.where(np.isnan(data), 0.0, data)
    return _ndz(filled, factor, order=order)

def _smooth_and_pad(da, out_step=0.25, pad_deg=2):
    """
    Resample DataArray to `out_step` degree grid, extending `pad_deg` beyond the data range.
    Uses np.pad(mode='edge') so the border cells carry the nearest real value.
    This guarantees no NaN at any output point.

    Steps
    -----
    1. sortby lat so array is S→N.
    2. np.pad the raw numpy array (edge-fill) by `n_pad` cells each side.
    3. Build matching padded coordinate arrays.
    4. xr.interp onto the fine output grid (all points are within the padded range, so interpolation never extrapolates → no NaN output).
    """
    da = da.sortby('lat')
    lats = da.lat.values.astype(float)
    lons = da.lon.values.astype(float)
    data = np.where(np.isnan(da.values.astype(float)),
                    np.nanmean(da.values), da.values.astype(float))

    lat_step = float(np.diff(lats).mean())
    lon_step = float(np.diff(lons).mean())
    n_pad_lat = max(1, int(np.ceil(pad_deg / lat_step)))
    n_pad_lon = max(1, int(np.ceil(pad_deg / lon_step)))

    # Pad the data array (edge mode replicates border values)
    data_pad = np.pad(data, ((n_pad_lat, n_pad_lat), (n_pad_lon, n_pad_lon)),
                      mode='edge')

    # Build padded coordinates
    lats_pad = np.concatenate([
        lats[0]  - np.arange(n_pad_lat, 0, -1) * lat_step,
        lats,
        lats[-1] + np.arange(1, n_pad_lat + 1)  * lat_step,
    ])
    lons_pad = np.concatenate([
        lons[0]  - np.arange(n_pad_lon, 0, -1) * lon_step,
        lons,
        lons[-1] + np.arange(1, n_pad_lon + 1)  * lon_step,
    ])

    da_pad = xr.DataArray(data_pad,
                          dims=['lat', 'lon'],
                          coords={'lat': lats_pad, 'lon': lons_pad})

    # Fine output grid
    fine_lats = np.arange(lats_pad[0],  lats_pad[-1]  + out_step, out_step)
    fine_lons = np.arange(lons_pad[0],  lons_pad[-1]  + out_step, out_step)

    return da_pad.interp(lat=fine_lats, lon=fine_lons, method='linear')


def _add_style(ax, im, unit, cbar_pad, shrink, ticks, label_size, extent,
               proj=ccrs.PlateCarree()):
    _add_ocean_mask(ax)
    ax.coastlines(resolution='10m', linewidth=0.8, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.6,
                   alpha=0.8, zorder=3)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linestyle='--',
                      linewidth=0.4, alpha=0.5, color='gray', zorder=4)
    gl.top_labels = False; gl.right_labels = False
    gl.xlabel_style = {'size': label_size - 1}
    gl.ylabel_style = {'size': label_size - 1}
    cbar = plt.colorbar(im, ax=ax, pad=cbar_pad, shrink=shrink, ticks=ticks)
    cbar.ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'{int(x)}'))
    cbar.ax.tick_params(labelsize=label_size - 1)
    return cbar


def plot_div(ax, da, vmin, vmax, extent, cmap='RdBu_r', center=0,
             cbar_pad=0.10, shrink=0.65, unit='mm/year',
             tick_step=None, label_size=9):
    norm   = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
    da_sm  = _smooth_and_pad(da)
    lons_f = da_sm.lon.values
    lats_f = da_sm.lat.values
    data_f = _upsample(da_sm.values, factor=4, order=1)
    lons_u = np.linspace(lons_f[0], lons_f[-1], data_f.shape[1])
    lats_u = np.linspace(lats_f[0], lats_f[-1], data_f.shape[0])
    im = ax.pcolormesh(lons_u, lats_u, data_f, cmap=cmap, norm=norm,
                       transform=ccrs.PlateCarree(),
                       shading='gouraud', zorder=1)
    if tick_step is None:
        span = vmax - vmin
        tick_step = next((s for s in [200,100,50,25,10] if span/s <= 12), 50)
    # Round vmin/vmax to tick_step multiples so ticks are always clean integers
    t_min = int(np.ceil(vmin  / tick_step)) * tick_step
    t_max = int(np.floor(vmax / tick_step)) * tick_step
    ticks = list(range(t_min, t_max + 1, tick_step))
    cbar  = _add_style(ax, im, unit, cbar_pad, shrink, ticks, label_size, extent)
    cbar.set_label(unit, size=label_size)
    return im


def plot_seq(ax, da, vmin, vmax, extent, cmap='Blues',
             cbar_pad=0.10, shrink=0.65, unit='mm/year',
             tick_step=None, label_size=9):
    norm   = mcolors.Normalize(vmin=vmin, vmax=vmax)
    da_sm  = _smooth_and_pad(da)
    lons_f = da_sm.lon.values
    lats_f = da_sm.lat.values
    data_f = _upsample(da_sm.values, factor=4, order=1)
    lons_u = np.linspace(lons_f[0], lons_f[-1], data_f.shape[1])
    lats_u = np.linspace(lats_f[0], lats_f[-1], data_f.shape[0])
    im = ax.pcolormesh(lons_u, lats_u, data_f, cmap=cmap, norm=norm,
                       transform=ccrs.PlateCarree(),
                       shading='gouraud', zorder=1)
    if tick_step is None:
        span = vmax - vmin
        tick_step = next((s for s in [100,50,25,10] if span/s <= 10), 25)
    t_min = int(np.ceil(vmin  / tick_step)) * tick_step
    t_max = int(np.floor(vmax / tick_step)) * tick_step
    ticks = list(range(t_min, t_max + 1, tick_step))
    cbar  = _add_style(ax, im, unit, cbar_pad, shrink, ticks, label_size, extent)
    cbar.set_label(unit, size=label_size)
    return im


def plot_stress(ax, da, extent, cbar_pad=0.10, shrink=0.65, label_size=9):
    cmap_ws = mcolors.ListedColormap(['#4CAF50', '#FFC107', '#F44336'])
    norm_ws = mcolors.BoundaryNorm([0.5, 1.5, 2.5, 3.5], ncolors=3)
    da_sm   = _smooth_and_pad(da)
    lons_f  = da_sm.lon.values
    lats_f  = da_sm.lat.values
    data_f  = _upsample(np.round(da_sm.values), factor=4, order=0)
    lons_u  = np.linspace(lons_f[0], lons_f[-1], data_f.shape[1])
    lats_u  = np.linspace(lats_f[0], lats_f[-1], data_f.shape[0])
    im = ax.pcolormesh(lons_u, lats_u, data_f, cmap=cmap_ws, norm=norm_ws,
                       transform=ccrs.PlateCarree(),
                       shading='nearest', zorder=1)
    _add_ocean_mask(ax)
    ax.coastlines(resolution='10m', linewidth=0.8, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.6,
                   alpha=0.8, zorder=3)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linestyle='--',
                      linewidth=0.4, alpha=0.5, color='gray', zorder=4)
    gl.top_labels = False; gl.right_labels = False
    gl.xlabel_style = {'size': label_size - 1}
    gl.ylabel_style = {'size': label_size - 1}
    cbar = plt.colorbar(im, ax=ax, pad=cbar_pad, shrink=shrink, ticks=[1,2,3])
    cbar.ax.set_yticklabels(
        ['Low\n(WSI ≥ −10%)', 'Moderate\n(−20% ≤ WSI < −10%)',
         'High\n(WSI < −20%)'],
        fontsize=label_size - 1)
    return im

print('Visualisation helpers ready.')


## Step 3: EQM function

In [ ]:
def eqm_monthly(gcm_base_mon, obs_mon, gcm_target_mon, n_quantiles=99):
    """
    Empirical Quantile Mapping (per calendar month, PCHIP interpolation)

    For each month 1-12:
      1. Build empirical CDFs from GCM baseline and obs at n_quantiles+2 points.
      2. Map each target value via PCHIP from GCM CDF to obs CDF.
      3. Clip out-of-range to [obs_min, obs_max]; clip negatives to 0.
    All inputs must share the same (lat, lon) grid.
    """
    q   = np.linspace(0, 1, n_quantiles + 2)
    out = []
    for month in range(1, 13):
        gb = gcm_base_mon.sel(  time=gcm_base_mon.time.dt.month   == month)
        ob = obs_mon.sel(        time=obs_mon.time.dt.month        == month)
        gt = gcm_target_mon.sel( time=gcm_target_mon.time.dt.month == month)

        gq = np.nanquantile(gb.values, q, axis=0)   # (nq, nlat, nlon)
        oq = np.nanquantile(ob.values, q, axis=0)

        tv   = gt.values.copy()
        cv   = np.empty_like(tv)
        nlat = tv.shape[1]; nlon = tv.shape[2]

        for i in range(nlat):
            for j in range(nlon):
                gc = gq[:, i, j]; oc = oq[:, i, j]
                if np.all(np.isnan(gc)) or np.all(np.isnan(oc)):
                    cv[:, i, j] = np.nan; continue
                _, idx = np.unique(gc, return_index=True)
                gc = gc[idx]; oc = oc[idx]
                pchip  = PchipInterpolator(gc, oc, extrapolate=False)
                mapped = pchip(tv[:, i, j])
                lo, hi = np.nanmin(oc), np.nanmax(oc)
                mapped = np.where(np.isnan(mapped) & (tv[:, i, j] < gc[0]),  lo, mapped)
                mapped = np.where(np.isnan(mapped) & (tv[:, i, j] > gc[-1]), hi, mapped)
                cv[:, i, j] = np.clip(mapped, 0, None)

        out.append(gt.copy(data=cv))

    result = xr.concat(out, dim='time').sortby('time')
    result.attrs.update({'long_name': 'Bias-corrected monthly precipitation',
                         'units': 'mm/month',
                         'bc_method': 'EQM monthly per-calendar-month PCHIP 99q'})
    return result

print('EQM function defined.')


## Step 4: Load CRU-TS observations

In [ ]:
print('Loading CRU-TS v4.08 monthly ...')
ds_cru = xr.open_dataset(CRU_FILE, chunks={'time': 120})
cvar   = 'pre' if 'pre' in ds_cru.data_vars else list(ds_cru.data_vars)[0]
obs_raw = (ds_cru[cvar]
           .sortby('lat')
           .sel(lon=RUSSIA_BOUNDS['lon'], lat=RUSSIA_BOUNDS['lat'],
                time=slice(BC_START, BC_END)))

print(f'  Variable : "{cvar}"')
print(f'  Shape    : {obs_raw.shape}')
print(f'  Lat      : {float(obs_raw.lat.min()):.2f} to {float(obs_raw.lat.max()):.2f}')
print(f'  Lon      : {float(obs_raw.lon.min()):.2f} to {float(obs_raw.lon.max()):.2f}')
print(f'  Mean     : {float(obs_raw.mean()):.2f} mm/month')


## Step 5: Bias correction pipeline (all 3 GCMs)

In [ ]:
def load_and_merge(paths):
    if isinstance(paths, str):
        return xr.open_dataset(paths, chunks={'time': 365})
    return xr.concat([xr.open_dataset(p, chunks={'time': 365})
                      for p in paths], dim='time')

bc_results = {}
print('=' * 62)
print('BIAS CORRECTION PIPELINE  —  RUSSIA')
print('=' * 62)

for model_name, minfo in models.items():
    print(f'\n--- {model_name}  (calendar: {minfo["calendar"]}) ---')
    try:
        ds_h = load_and_merge(minfo['historical'])
        ds_r = load_and_merge(minfo['rcp45'])
        ds_h['pr_mm'] = ds_h['pr'] * 86400
        ds_r['pr_mm'] = ds_r['pr'] * 86400

        hist_ind = ds_h['pr_mm'].sel(**RUSSIA_BOUNDS)
        rcp_ind  = ds_r['pr_mm'].sel(**RUSSIA_BOUNDS)

        hist_mon = hist_ind.resample(time='MS').sum()
        rcp_mon  = rcp_ind.resample(time='MS').sum()

        gcm_base = hist_mon.sel(time=slice(BC_START, BC_END)).load()
        gcm_fut  = rcp_mon.sel( time=slice(FUT_START, FUT_END)).load()
        print(f'  GCM grid : {len(gcm_base.lat)} lat x {len(gcm_base.lon)} lon')
        print(f'  Baseline : {len(gcm_base.time)} months  |  Future: {len(gcm_fut.time)} months')

        # Regrid CRU-TS → GCM grid (coarse; EQM operates at GCM resolution)
        obs_gcm = obs_raw.interp(lat=gcm_base.lat, lon=gcm_base.lon,
                                 method='linear').load()

        print(f'  Running EQM ...')
        pr_bc_base = eqm_monthly(gcm_base, obs_gcm, gcm_base, N_QUANTILES)
        pr_bc_fut  = eqm_monthly(gcm_base, obs_gcm, gcm_fut,  N_QUANTILES)

        prcptot_base = pr_bc_base.resample(time='YE').sum()
        prcptot_fut  = pr_bc_fut.resample( time='YE').sum()
        base_mean    = prcptot_base.mean(dim='time')
        fut_mean     = prcptot_fut.mean( dim='time')
        anomaly      = fut_mean - base_mean

        bc_results[model_name] = {
            'baseline_mean':       base_mean,
            'future_mean':         fut_mean,
            'anomaly':             anomaly,
            'prcptot_bc_baseline': prcptot_base,
            'prcptot_bc_future':   prcptot_fut,
            'pr_bc_baseline':      pr_bc_base,
            'pr_bc_future':        pr_bc_fut,
        }
        print(f'    done  |  base={float(base_mean.mean()):.1f}  '
              f'fut={float(fut_mean.mean()):.1f}  '
              f'anom={float(anomaly.mean()):.1f} mm/yr')

    except Exception as e:
        import traceback
        print(f'    ERROR: {e}'); traceback.print_exc()

print(f'\nProcessed {len(bc_results)} / {len(models)} models.')


## Step 6: Save per-model NetCDFs

In [ ]:
print(f'Saving per-model NetCDFs \u2192 {BC_DIR}')
for mn, d in bc_results.items():
    safe = mn.replace('-', '_')
    # Annual PRCPTOT (baseline & future)
    d['prcptot_bc_baseline'].to_netcdf(
        os.path.join(BC_DIR, f'PRCPTOT_bc_baseline_{safe}_Russia.nc'))
    d['prcptot_bc_future'].to_netcdf(
        os.path.join(BC_DIR, f'PRCPTOT_bc_future_{safe}_Russia.nc'))
    # Monthly BC precipitation (baseline & future)
    d['pr_bc_baseline'].to_netcdf(
        os.path.join(BC_DIR, f'pr_bc_baseline_{safe}_Russia.nc'))
    d['pr_bc_future'].to_netcdf(
        os.path.join(BC_DIR, f'pr_bc_future_{safe}_Russia.nc'))
    print(f'  - {mn}  (annual + monthly)')
print('Done.')


## Step 7: Ensemble statistics & CWSI

In [ ]:
if len(bc_results) == 0:
    raise RuntimeError('No models bias-corrected. Check errors above.')

ref_model = list(bc_results.keys())[0]
ref_grid  = bc_results[ref_model]['baseline_mean']
print(f'Reference grid: {ref_model}  '
      f'({len(ref_grid.lat)} lat x {len(ref_grid.lon)} lon)')

# Align all models to reference grid
regrid = {}
for mn, d in bc_results.items():
    regrid[mn] = {
        k: d[k].interp(lat=ref_grid.lat, lon=ref_grid.lon, method='nearest')
        for k in ('baseline_mean', 'future_mean', 'anomaly')
    }

baseline_stack = xr.concat([regrid[m]['baseline_mean'] for m in regrid], dim='model')
future_stack   = xr.concat([regrid[m]['future_mean']   for m in regrid], dim='model')
anomaly_stack  = xr.concat([regrid[m]['anomaly']       for m in regrid], dim='model')

ens_base = baseline_stack.mean(dim='model')
ens_fut  = future_stack.mean(  dim='model')
ens_anom = anomaly_stack.mean( dim='model')
ens_std  = anomaly_stack.std(  dim='model')

# WSI, relative change in percent
ens_wsi = xr.where(ens_base > 0,
                   (ens_fut - ens_base) / ens_base * 100,
                   np.nan)

# CWSI stress classification
stress = xr.full_like(ens_wsi, fill_value=np.nan)
stress = xr.where(ens_wsi >= -10,                       1.0, stress)  # Low
stress = xr.where((ens_wsi < -10) & (ens_wsi >= -20),  2.0, stress)  # Moderate
stress = xr.where(ens_wsi < -20,                        3.0, stress)  # High

# Count only valid (non-NaN) land pixels for percentages
s_vals   = stress.values.ravel()
valid    = s_vals[~np.isnan(s_vals)]
n_valid  = len(valid)
low_pct  = np.sum(valid == 1) / n_valid * 100 if n_valid else 0
mod_pct  = np.sum(valid == 2) / n_valid * 100 if n_valid else 0
high_pct = np.sum(valid == 3) / n_valid * 100 if n_valid else 0

print('\n' + '=' * 57)
print('ENSEMBLE STATISTICS — Russia (Bias-Corrected)')
print('=' * 57)
print(f'  BC Baseline mean PRCPTOT : {float(ens_base.mean()):.1f} mm/yr')
print(f'  BC Future   mean PRCPTOT : {float(ens_fut.mean()):.1f} mm/yr')
print(f'  Ensemble anomaly mean    : {float(ens_anom.mean()):.1f} mm/yr')
print(f'  Ensemble anomaly std     : {float(ens_std.mean()):.1f} mm/yr')
print(f'  Mean WSI                 : {float(ens_wsi.mean()):.1f} %')
print(f'\n  Low stress    (WSI >= -10%)        : {low_pct:.1f}%')
print(f'  Moderate      (-20% <= WSI < -10%) : {mod_pct:.1f}%')
print(f'  High stress   (WSI < -20%)         : {high_pct:.1f}%')


## Step 8: Save ensemble NetCDFs

In [ ]:
for fname, da in {
    'Ensemble_Anomaly_Russia.nc':     ens_anom,
    'Ensemble_Uncertainty_Russia.nc': ens_std,
    'Ensemble_WaterStress_Russia.nc': stress,
    'Ensemble_WSI_Russia.nc':         ens_wsi,
}.items():
    da.to_netcdf(os.path.join(ENS_DIR, fname))
    print(f'  Exported: {fname}')
print(f'Ensemble NetCDFs saved → {ENS_DIR}')


## Step 9: Multi-model ensemble map

In [ ]:
fig = plt.figure(figsize=(24, 12))
fig.suptitle(
    'Russia — Bias-Corrected PRCPTOT (RCP 4.5, 2041–2070 vs 1976–2005)\n'
    'EQM · CRU-TS v4.08 · 3-GCM Ensemble',
    fontsize=13, fontweight='bold', y=1.01)

for i, mn in enumerate(regrid.keys()):
    ax = fig.add_subplot(3, 3, i+1, projection=RUSSIA_PROJ)
    plot_div(ax, regrid[mn]['anomaly'],
             vmin=-160, vmax=160, extent=RUSSIA_EXTENT,
             tick_step=40, unit='mm/year', shrink=0.42)
    ax.set_title(f'{mn}\nAnomaly (BC)', fontsize=10, fontweight='bold')

ax4 = fig.add_subplot(3, 3, 4, projection=RUSSIA_PROJ)
plot_div(ax4, ens_anom, vmin=-160, vmax=160, extent=RUSSIA_EXTENT,
         tick_step=40, unit='mm/year', shrink=0.42)
ax4.set_title('Ensemble Mean Anomaly (BC)', fontsize=10, fontweight='bold')

ax5 = fig.add_subplot(3, 3, 5, projection=RUSSIA_PROJ)
plot_seq(ax5, ens_std, vmin=0, vmax=150, extent=RUSSIA_EXTENT,
         cmap='YlOrRd', tick_step=25, unit='mm/year', shrink=0.42)
ax5.set_title('Model Uncertainty — Std Dev (BC)', fontsize=10, fontweight='bold')

ax6 = fig.add_subplot(3, 3, 6, projection=RUSSIA_PROJ)
plot_stress(ax6, stress, extent=RUSSIA_EXTENT, shrink=0.42)
ax6.set_title('Water Stress CWSI (BC Ensemble)', fontsize=10, fontweight='bold')

ax7 = fig.add_subplot(3, 3, 7, projection=RUSSIA_PROJ)
pct_inc = (anomaly_stack > 0).sum(dim='model') / len(bc_results) * 100
plot_div(ax7, pct_inc, vmin=0, vmax=100, extent=RUSSIA_EXTENT,
         center=50, tick_step=25, unit='% Models', shrink=0.42)
ax7.set_title('% Models Showing Precipitation Increase',
              fontsize=10, fontweight='bold')

plt.tight_layout(pad=2.0, w_pad=3.0, h_pad=3.5)
out = os.path.join(ENS_DIR, 'BC_MultiModel_Russia_Analysis.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out}')


## Step 10: Validation time series

In [ ]:
cru_ann  = obs_raw.resample(time='YE').sum().mean(dim=['lat','lon'])
colors_m = {'HadGEM2-AO': '#1f77b4', 'MPI-ESM-MR': '#ff7f0e',
             'IPSL-CM5A-LR': '#2ca02c'}

fig, axes = plt.subplots(1, len(bc_results),
                          figsize=(7*len(bc_results), 5), sharey=False)
if len(bc_results) == 1: axes = [axes]

for ax, (mn, data) in zip(axes, bc_results.items()):
    bc_ts = data['prcptot_bc_baseline'].mean(dim=['lat','lon'])
    ax.plot(cru_ann.time.dt.year.values, cru_ann.values,
            color='black', lw=2, label='CRU-TS observed', zorder=3)
    ax.plot(bc_ts.time.dt.year.values, bc_ts.values,
            color=colors_m.get(mn,'steelblue'), lw=1.5,
            ls='--', alpha=0.85, label=f'{mn} BC', zorder=2)
    ax.axhline(float(cru_ann.mean()), color='black', lw=1, ls=':', alpha=0.5)
    ax.axhline(float(bc_ts.mean()),
               color=colors_m.get(mn,'steelblue'), lw=1, ls=':', alpha=0.5)
    ax.set_title(f'{mn}\nBC baseline vs CRU-TS (1976–2005)',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Year'); ax.set_ylabel('PRCPTOT (mm/yr)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

fig.suptitle('Russia — BC Validation: Annual PRCPTOT Area Mean',
             fontsize=12, fontweight='bold')
plt.tight_layout()
out = os.path.join(ENS_DIR, 'BC_Validation_Russia.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out}')


## Step 11: Summary CSV

In [ ]:
rows = []
for mn, d in bc_results.items():
    bm = float(d['baseline_mean'].mean())
    fm = float(d['future_mean'].mean())
    am = float(d['anomaly'].mean())
    rows.append({'Region':'Russia','Model':mn,'Type':'Individual (BC)',
                 'BC_Baseline_mm':round(bm,2),'BC_Future_mm':round(fm,2),
                 'BC_Anomaly_mm':round(am,2),
                 'Change_pct':round(am/bm*100,2) if bm else np.nan})

bm_e=float(ens_base.mean()); fm_e=float(ens_fut.mean()); am_e=float(ens_anom.mean())
rows.append({'Region':'Russia','Model':'Ensemble Mean','Type':'Ensemble (BC)',
             'BC_Baseline_mm':round(bm_e,2),'BC_Future_mm':round(fm_e,2),
             'BC_Anomaly_mm':round(am_e,2),
             'Change_pct':round(am_e/bm_e*100,2) if bm_e else np.nan})

df  = pd.DataFrame(rows)
out = os.path.join(ENS_DIR, 'BC_Summary_Russia.csv')
df.to_csv(out, index=False)
print(df.to_string(index=False))
print(f'\nSaved: {out}')


## Step 12: Cleanup

In [ ]:
client.close()
print('Dask client closed.')
print('\n' + '=' * 55)
print('  BIAS CORRECTION  —  COMPLETE  ')
print('=' * 55)
